In [1]:
# Project 3: Data Cleaning & Preparation
# Framework: Transforming Raw Datasets into Analysis-Ready Assets

In [7]:
import os
import pandas as pd
import numpy as np

file_path = "hotel_bookings.csv"

if not os.path.exists(file_path):
    file_path = "hotel_bookings.xlsx"

if file_path.endswith('.csv'):
    df = pd.read_csv(file_path)
else:
    df = pd.read_excel(file_path)

df.head(5)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## Handling missing values, duplicates, & inconsistencies

In [8]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

df = df.drop_duplicates()

if 'children' in df.columns:
    df['children'] = df['children'].fillna(0)
if 'country' in df.columns:
    df['country'] = df['country'].fillna('Unknown')
if 'agent' in df.columns:
    df['agent'] = df['agent'].fillna(0)
if 'company' in df.columns:
    df['company'] = df['company'].fillna(0)

df['total_guests'] = df['adults'] + df['children'] + df['babies']
df = df[df['total_guests'] > 0]

## raw data for analysis

In [9]:
df['total_stay_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['estimated_revenue'] = df['total_stay_nights'] * df['adr']

def categorize_lead_time(days):
    if days <= 7:
        return 'Last Minute'
    elif days <= 30:
        return 'Short Lead'
    elif days <= 90:
        return 'Medium Lead'
    else:
        return 'Long Lead'

df['lead_time_category'] = df['lead_time'].apply(categorize_lead_time)
df['is_canceled_label'] = df['is_canceled'].map({0: 'Check-Out', 1: 'Canceled'})

df['arrival_date_full'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' + 
    df['arrival_date_month'] + '-' + 
    df['arrival_date_day_of_month'].astype(str),
    errors='coerce'
)

df[['hotel', 'lead_time_category', 'total_stay_nights', 'estimated_revenue', 'is_canceled_label']].head()

,hotel,lead_time_category,total_stay_nights,estimated_revenue,is_canceled_label
0,Resort Hotel,Long Lead,0,0.0,Check-Out
1,Resort Hotel,Long Lead,0,0.0,Check-Out
2,Resort Hotel,Last Minute,1,75.0,Check-Out
3,Resort Hotel,Short Lead,1,75.0,Check-Out
4,Resort Hotel,Short Lead,2,196.0,Check-Out


## clean, analysis-ready data

In [10]:
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

df.to_csv(os.path.join(output_dir, "clean_hotel_bookings.csv"), index=False)

summary_report = df.groupby(['hotel', 'is_canceled_label']).agg(
    total_bookings=('hotel', 'count'),
    total_revenue=('estimated_revenue', 'sum'),
    avg_daily_rate=('adr', 'mean')
).reset_index()

with pd.ExcelWriter(os.path.join(output_dir, "Analysis_Ready_Hotel_Report.xlsx"), engine='openpyxl') as writer:
    summary_report.to_excel(writer, sheet_name='Hotel Summary', index=False)

display(summary_report)

,hotel,is_canceled_label,total_bookings,total_revenue,avg_daily_rate
0,City Hotel,Canceled,16035,6614455.63,117.379025
1,City Hotel,Check-Out,37239,12150577.73,108.642289
2,Resort Hotel,Canceled,7974,4869587.35,118.799984
3,Resort Hotel,Check-Out,25982,10816970.42,93.001073
